# Initial Script to get base data for finetuning


In [ ]:
import json
import re
from pathlib import Path
from random import shuffle
from datasets import load_dataset

# CONFIG

PROJECT_ROOT = Path.cwd()
OUTPUT_PATH = PROJECT_ROOT / "data" / "sft.jsonl"
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

MAX_ULTRACHAT = 200000
MAX_OPENHERMES = 200000

MIN_SCORE = 6
MIN_RESPONSE_WORDS = 45
SEED = 42

# REGEX HELPERS

def compile_patterns(words):
    return [re.compile(rf"\b{re.escape(w)}\b", re.IGNORECASE) for w in words]

def regex_hits(text, patterns):
    return sum(1 for p in patterns if p.search(text))

def normalize(text: str) -> str:
    return re.sub(r"\s+", " ", text.lower()).strip()

# DOMAIN SIGNALS

# --- Core lending concepts ---
LENDING_OBJECTS = compile_patterns([
    "loan", "credit", "debt", "mortgage",
    "interest", "apr", "repayment",
    "installment", "balance"
])

LENDING_ACTIONS = compile_patterns([
    "approve", "deny", "qualify", "assess",
    "evaluate", "determine", "repay",
    "default", "borrow", "lend", "charge"
])

LENDING_ENTITIES = compile_patterns([
    "borrower", "lender", "bank",
    "creditor", "applicant", "consumer"
])

DECISION_PHRASES = compile_patterns([
    "what happens if",
    "how might",
    "under what conditions",
    "why do lenders",
    "can a borrower",
    "would this affect",
    "how does this affect"
])

# --- Structural consequence ---
CONSEQUENCES = compile_patterns([
    "interest rate", "approval", "rejection",
    "repayment", "default", "risk", "terms"
])

# --- Blacklist ---
BLACKLIST = compile_patterns([
    "job", "career", "resume", "human resources",
    "temporary work", "employment",
    "history of", "curriculum",
    "spreadsheet", "excel",
    "blockchain technology",
    "political", "government policy"
])

GENERIC_OPENERS = [
    re.compile(r"^what is\b"),
    re.compile(r"^define\b"),
    re.compile(r"^explain\b"),
]

# SCORING + STRUCTURE

def lending_score(inst: str, resp: str) -> int:
    text = inst + " " + resp
    score = 0
    score += regex_hits(text, LENDING_OBJECTS) * 3
    score += regex_hits(text, LENDING_ACTIONS) * 2
    score += regex_hits(text, LENDING_ENTITIES) * 2
    score += regex_hits(inst, DECISION_PHRASES) * 3
    return score

def generic_penalty(inst: str) -> int:
    return 2 if any(p.match(inst) for p in GENERIC_OPENERS) else 0

def is_blacklisted(text: str) -> bool:
    return regex_hits(text, BLACKLIST) > 0

def has_lending_structure(inst: str, resp: str) -> bool:
    """
    HARD REQUIREMENT:
    Must contain:
      - an ACTOR
      - a DECISION/EVALUATION
      - a CONSEQUENCE
    """
    text = inst + " " + resp
    return (
        regex_hits(text, LENDING_ENTITIES) > 0 and
        regex_hits(text, LENDING_ACTIONS) > 0 and
        regex_hits(text, CONSEQUENCES) > 0
    )

# DATA EXTRACTION

def extract_ultrachat():
    ds = load_dataset("HuggingFaceH4/ultrachat_200k", split="train_sft")
    rows = ds.shuffle(seed=SEED).select(range(MAX_ULTRACHAT))
    samples = []

    for r in rows:
        msgs = r["messages"]
        if len(msgs) < 2:
            continue
        if msgs[0]["role"] != "user" or msgs[1]["role"] != "assistant":
            continue
        samples.append({
            "instruction": msgs[0]["content"],
            "response": msgs[1]["content"],
            "source": "ultrachat"
        })

    return samples


def extract_openhermes():
    ds = load_dataset("teknium/OpenHermes-2.5", split="train")
    rows = ds.shuffle(seed=SEED).select(range(MAX_OPENHERMES))
    samples = []

    for r in rows:
        conv = r.get("conversations") or r.get("messages")
        if not conv or len(conv) < 2:
            continue

        if "from" in conv[0]:
            if conv[0]["from"] != "human":
                continue
            inst = conv[0]["value"]
            resp = conv[1]["value"]
        elif "role" in conv[0]:
            if conv[0]["role"] != "user":
                continue
            inst = conv[0]["content"]
            resp = conv[1]["content"]
        else:
            continue

        samples.append({
            "instruction": inst,
            "response": resp,
            "source": "openhermes"
        })

    return samples

# FILTER PIPELINE

def filter_samples(samples):
    kept = []

    for s in samples:
        inst = normalize(s["instruction"])
        resp = normalize(s["response"])
        text = inst + " " + resp

        # Hard rejections
        if is_blacklisted(text):
            continue
        if len(resp.split()) < MIN_RESPONSE_WORDS:
            continue
        if not (
            regex_hits(text, LENDING_ENTITIES) > 0 and
            regex_hits(text, LENDING_ACTIONS) > 0 and
            (
                regex_hits(text, CONSEQUENCES) > 0
                or regex_hits(inst, DECISION_PHRASES) > 0
            )
        ):
            continue


        # Scoring
        score = lending_score(inst, resp)
        score -= generic_penalty(inst)

        if score < MIN_SCORE:
            continue

        kept.append({
            "instruction": s["instruction"].strip(),
            "response": s["response"].strip(),
            "_meta": {
                "source": s["source"],
                "score": score
            }
        })

    return kept

# MAIN

def main():
    print("Loading UltraChat…")
    uc = extract_ultrachat()
    print(f"UltraChat candidates: {len(uc)}")

    print("Loading OpenHermes…")
    oh = extract_openhermes()
    print(f"OpenHermes candidates: {len(oh)}")

    combined = uc + oh
    shuffle(combined)

    print("Filtering with structural constraints…")
    filtered = filter_samples(combined)
    print(f"Final kept: {len(filtered)}")

    with OUTPUT_PATH.open("w") as f:
        for s in filtered:
            f.write(json.dumps(
                {"instruction": s["instruction"], "response": s["response"]},
                ensure_ascii=False
            ) + "\n")

    print(f"Dataset written to {OUTPUT_PATH}")

if __name__ == "__main__":
    main()


### Script that takes existing gold samples (made with GPT-5.2 manuallly) and uses them to create prompts and then batch sends those prompts to make more synthetic samples

In [ ]:
import json
from pathlib import Path

GOLD_PATH = PROJECT_ROOT / "gold.jsonl"

SYNTHETIC_PER_GOLD = 3

def load_gold():
    with GOLD_PATH.open() as f:
        return [json.loads(l) for l in f]

def build_prompt(sample):
    return f"""
You are generating training data for a lending decision assistant.

Below is a gold-standard example.

INSTRUCTION:
{sample["instruction"]}

RESPONSE:
{sample["response"]}

Generate ONE new example that:
- keeps the same loan type
- keeps the same decision type (approval, denial, or interest rate)
- changes the numeric values (income, debt, credit score, loan amount, term)
- remains realistic and internally consistent
- does NOT introduce new concepts or rules

Output ONLY valid JSON:
{{"instruction": "...", "response": "..."}}
""".strip()

def main():
    gold = load_gold()
    prompts = []

    for g in gold:
        for _ in range(SYNTHETIC_PER_GOLD):
            prompts.append(build_prompt(g))

    # Write prompts for manual or API-based generation
    Path("data/synthetic_prompts.txt").write_text(
        "\n\n---\n\n".join(prompts)
    )

    print(f"Generated {len(prompts)} synthetic prompts.")

if __name__ == "__main__":
    main()


In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()  # loads .env from current working directory

print(os.getenv("OPENAI_API_KEY"))

In [ ]:
import os
import json
import time
import math
from pathlib import Path
from typing import List

from openai import OpenAI
from openai import RateLimitError, APIError, APIConnectionError

# CONFIG

PROMPTS_PATH = Path("data/synthetic_prompts.txt")
RAW_OUT_PATH = Path("data/synthetic_raw.jsonl")

MODEL = "gpt-4.1-mini"
TEMPERATURE = 0.3
MAX_TOKENS = 450

BASE_DELAY = 1.0          # base delay between requests (seconds)
MAX_RETRIES = 5           # per prompt
BACKOFF_MULTIPLIER = 2.0  # exponential backoff

# HELPERS

def load_prompts() -> List[str]:
    text = PROMPTS_PATH.read_text()
    return [p.strip() for p in text.split("\n\n---\n\n") if p.strip()]

def already_generated() -> int:
    if not RAW_OUT_PATH.exists():
        return 0
    return sum(1 for _ in RAW_OUT_PATH.open())

def parse_json_safe(text: str):
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return None

# MAIN

def main():
    api_key = os.getenv("OPENAI_API_KEY")
    if not api_key:
        raise RuntimeError("OPENAI_API_KEY not found in environment")

    client = OpenAI(api_key=api_key)

    prompts = load_prompts()
    completed = already_generated()

    print(f"Loaded {len(prompts)} total prompts")
    print(f"Already completed: {completed}")

    prompts = prompts[completed:]
    total = completed + len(prompts)

    if not prompts:
        print("Nothing left to generate.")
        return

    RAW_OUT_PATH.parent.mkdir(parents=True, exist_ok=True)

    with RAW_OUT_PATH.open("a") as out:
        for idx, prompt in enumerate(prompts, start=completed + 1):
            print(f"\n[{idx}/{total}] Generating synthetic example")

            attempt = 0
            delay = BASE_DELAY

            while attempt <= MAX_RETRIES:
                try:
                    response = client.responses.create(
                        model=MODEL,
                        input=prompt,
                        temperature=TEMPERATURE,
                        max_output_tokens=MAX_TOKENS,
                    )

                    text = response.output_text
                    parsed = parse_json_safe(text)

                    record = {
                        "prompt_id": idx,
                        "prompt": prompt,
                        "raw_output": text,
                        "parsed": parsed,
                    }

                    out.write(json.dumps(record, ensure_ascii=False) + "\n")
                    out.flush()

                    time.sleep(BASE_DELAY)
                    break  # success → exit retry loop

                except RateLimitError as e:
                    attempt += 1
                    print(f"Rate limit hit (attempt {attempt}/{MAX_RETRIES}). Backing off {delay:.1f}s")
                    time.sleep(delay)
                    delay *= BACKOFF_MULTIPLIER

                except APIConnectionError as e:
                    attempt += 1
                    print(f"Connection error (attempt {attempt}/{MAX_RETRIES}). Retrying in {delay:.1f}s")
                    time.sleep(delay)
                    delay *= BACKOFF_MULTIPLIER

                except APIError as e:
                    # This includes insufficient_quota
                    print("API error:", e)

                    if "insufficient_quota" in str(e).lower():
                        print("\nQuota exhausted. Stopping cleanly.")
                        print("All completed generations are safely saved.")
                        return

                    attempt += 1
                    time.sleep(delay)
                    delay *= BACKOFF_MULTIPLIER

                except Exception as e:
                    print("Unexpected error:", e)
                    print("Stopping to avoid corruption.")
                    return

            else:
                print("Max retries exceeded for this prompt. Skipping.")

    print("\nSynthetic generation finished successfully")

if __name__ == "__main__":
    main()


In [ ]:
import json
from pathlib import Path

# RAW_PATH = Path("synthetic_raw.jsonl")
# OUT_PATH = Path("synthetic_clean.jsonl")

# RAW_PATH = Path("denial_synthetic_raw.jsonl")
# OUT_PATH = Path("denial_synthetic_clean.jsonl")

RAW_PATH = Path("denial_synthetic_raw_v2.jsonl")
OUT_PATH = Path("denial_synthetic_clean_v2.jsonl")

# Heuristic validators

MIN_RESPONSE_WORDS = 80
MAX_RESPONSE_WORDS = 180

REQUIRED_TERMS = [
    "borrower",
    "lender"
]

DECISION_TERMS = [
    "approve", "approval",
    "deny", "denial",
    "interest rate",
    "qualify", "qualification"
]

BANNED_PHRASES = [
    "this is not financial advice",
    "consult a professional",
    "depends on many factors",
    "for educational purposes",
]

def is_valid_sample(sample: dict) -> bool:
    if not isinstance(sample, dict):
        return False

    inst = sample.get("instruction", "").strip().lower()
    resp = sample.get("response", "").strip().lower()

    if not inst or not resp:
        return False

    # basic structure
    if len(resp.split()) < MIN_RESPONSE_WORDS:
        return False
    if len(resp.split()) > MAX_RESPONSE_WORDS:
        return False

    # borrower + lender presence
    if not all(term in inst or term in resp for term in REQUIRED_TERMS):
        return False

    # decision signal
    if not any(term in inst or term in resp for term in DECISION_TERMS):
        return False

    # ban disclaimers / hedging
    if any(bad in resp for bad in BANNED_PHRASES):
        return False

    return True

# Main extraction

def main():
    kept = []
    rejected = 0
    invalid_json = 0

    with RAW_PATH.open() as f:
        for line in f:
            obj = json.loads(line)
            parsed = obj.get("parsed")

            if not parsed:
                invalid_json += 1
                continue

            if is_valid_sample(parsed):
                kept.append({
                    "instruction": parsed["instruction"].strip(),
                    "response": parsed["response"].strip()
                })
            else:
                rejected += 1

    OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
    with OUT_PATH.open("w") as out:
        for k in kept:
            out.write(json.dumps(k, ensure_ascii=False) + "\n")

    print("=== Synthetic extraction summary ===")
    print(f"Total raw entries     : {invalid_json + rejected + len(kept)}")
    print(f"Invalid / unparsed    : {invalid_json}")
    print(f"Rejected by filters   : {rejected}")
    print(f"Accepted synthetics: {len(kept)}")
    print(f"Written to            : {OUT_PATH}")

if __name__ == "__main__":
    main()


# Statistics of final set

In [ ]:
import json
from pathlib import Path
from statistics import mean, median

path = Path("sft.jsonl")
samples = [json.loads(l) for l in path.read_text().splitlines()]

print("Total samples:", len(samples))

inst_lens = [len(s["instruction"].split()) for s in samples]
resp_lens = [len(s["response"].split()) for s in samples]

print("\nInstruction length (words):")
print("  mean:", round(mean(inst_lens), 1))
print("  median:", median(inst_lens))
print("  min:", min(inst_lens))
print("  max:", max(inst_lens))

print("\nResponse length (words):")
print("  mean:", round(mean(resp_lens), 1))
print("  median:", median(resp_lens))
print("  min:", min(resp_lens))
print("  max:", max(resp_lens))

In [ ]:
from collections import Counter

def classify_decision(text: str):
    t = text.lower()
    if "deny" in t or "denial" in t or "decline" in t:
        return "denial"
    if "approve" in t or "approval" in t or "qualify" in t:
        return "approval"
    if "interest rate" in t or "higher rate" in t or "pricing" in t:
        return "pricing"
    return "other"

decision_counts = Counter(
    classify_decision(s["response"]) for s in samples
)

print(decision_counts)


In [ ]:
import re

def count_numbers(text):
    return len(re.findall(r"\d", text))

num_counts = [count_numbers(s["instruction"] + " " + s["response"]) for s in samples]

print("Numeric density:")
print("  mean digits per sample:", round(mean(num_counts), 1))
print("  min:", min(num_counts))
print("  max:", max(num_counts))


In [ ]:
loan_keywords = {
    "mortgage": ["mortgage", "refinance"],
    "auto": ["auto", "car"],
    "personal": ["personal loan", "unsecured"],
    "secured": ["secured", "collateral"],
    "credit": ["credit line", "credit card"],
    "business": ["business", "commercial"]
}

from collections import defaultdict

loan_counts = defaultdict(int)

for s in samples:
    text = (s["instruction"] + " " + s["response"]).lower()
    for k, kws in loan_keywords.items():
        if any(kw in text for kw in kws):
            loan_counts[k] += 1

loan_counts


In [ ]:
import re

dtis = []

for s in samples:
    matches = re.findall(r"(\d{2})\s*%", s["response"])
    for m in matches:
        dti = int(m)
        if 10 <= dti <= 70:
            dtis.append(dti)

print("DTI stats:")
print("  count:", len(dtis))
print("  min:", min(dtis))
print("  max:", max(dtis))
print("  mean:", round(mean(dtis), 1))


In [ ]:
def starts_with(text, phrase):
    return text.lower().startswith(phrase)

starts = Counter(
    starts_with(s["response"], "the lender would")
    for s in samples
)

starts


# Script to make corrections ot data set by adding more denials

In [ ]:
import os
import json
import time
from pathlib import Path
from typing import List

from openai import OpenAI
from openai import RateLimitError, APIError, APIConnectionError

# CONFIG

DATASET_PATH = Path("sft.jsonl")

# PROMPTS_OUT = Path("denial_prompts.txt")
# RAW_OUT = Path("denial_synthetic_raw.jsonl")
PROMPTS_OUT = Path("denial_prompts_v2.txt")
RAW_OUT = Path("denial_synthetic_raw_v2.jsonl")

MODEL = "gpt-4.1-mini"
TEMPERATURE = 0.3
MAX_TOKENS = 450

# DENIALS_TO_GENERATE = 35
DENIALS_TO_GENERATE = 70
BASE_DELAY = 1.0
MAX_RETRIES = 5
BACKOFF = 2.0

# FILTERING LOGIC

def is_high_risk(sample: dict) -> bool:
    text = (sample["instruction"] + " " + sample["response"]).lower()
    return (
        "unsecured" in text
        or "dt" in text and any(p in text for p in ["50%", "55%", "60%"])
        or "credit score" in text and any(s in text for s in ["620", "630", "640"])
        or "delinquen" in text
    )

def is_already_denial(sample: dict) -> bool:
    return any(
        t in sample["response"].lower()
        for t in ["deny", "denial", "decline", "not approve"]
    )

# PROMPT BUILDER

def build_denial_prompt(sample: dict) -> str:
    return f"""
You are generating training data for a lending decision assistant.

Below is a reference example.

INSTRUCTION:
{sample["instruction"]}

RESPONSE:
{sample["response"]}

Generate ONE new example that represents a FAILED loan application.

STRICT REQUIREMENTS (DO NOT VIOLATE):
- The lender MUST deny the loan application
- The response MUST clearly state that the loan is denied or declined
- Do NOT use conditional language (no "may", "might", "could")
- Do NOT suggest approval with changes
- Do NOT suggest pricing alternatives
- Do NOT provide advice to the borrower

REASONING REQUIREMENTS:
- Increase risk factors (higher DTI, lower credit score, or weaker repayment capacity)
- Mention at least TWO concrete risk factors
- Explain why denial is the lender’s final decision

STYLE:
- Professional, neutral lender tone
- 80–130 words
- No definitions, no policies, no disclaimers

Output ONLY valid JSON:
{{"instruction": "...", "response": "..."}}
""".strip()

# MAIN

def main():
    api_key = os.getenv("OPENAI_API_KEY")
    if not api_key:
        raise RuntimeError("OPENAI_API_KEY not found")

    client = OpenAI(api_key=api_key)

    samples = [json.loads(l) for l in DATASET_PATH.read_text().splitlines()]

    anchors = [
        s for s in samples
        if is_high_risk(s) and not is_already_denial(s)
    ]

    if not anchors:
        raise RuntimeError("No suitable high-risk anchors found")

    anchors = anchors[:DENIALS_TO_GENERATE]

    # Write prompts

    prompts = [build_denial_prompt(a) for a in anchors]

    PROMPTS_OUT.write_text("\n\n---\n\n".join(prompts))
    print(f"Wrote {len(prompts)} denial prompts")

    # Generate via API

    RAW_OUT.parent.mkdir(parents=True, exist_ok=True)
    completed = RAW_OUT.exists() and sum(1 for _ in RAW_OUT.open()) or 0

    with RAW_OUT.open("a") as out:
        for i, prompt in enumerate(prompts[completed:], start=completed + 1):
            attempt = 0
            delay = BASE_DELAY

            print(f"[{i}/{len(prompts)}] Generating denial sample")

            while attempt <= MAX_RETRIES:
                try:
                    resp = client.responses.create(
                        model=MODEL,
                        input=prompt,
                        temperature=TEMPERATURE,
                        max_output_tokens=MAX_TOKENS,
                    )

                    record = {
                        "prompt_id": i,
                        "prompt": prompt,
                        "raw_output": resp.output_text,
                        "parsed": None
                    }

                    try:
                        record["parsed"] = json.loads(resp.output_text)
                    except json.JSONDecodeError:
                        pass

                    out.write(json.dumps(record, ensure_ascii=False) + "\n")
                    out.flush()
                    time.sleep(BASE_DELAY)
                    break

                except RateLimitError:
                    attempt += 1
                    time.sleep(delay)
                    delay *= BACKOFF

                except APIConnectionError:
                    attempt += 1
                    time.sleep(delay)
                    delay *= BACKOFF

                except APIError as e:
                    if "insufficient_quota" in str(e).lower():
                        print("🚨 Quota exhausted, stopping safely.")
                        return
                    attempt += 1
                    time.sleep(delay)

    print("Denial synthetic generation complete")

if __name__ == "__main__":
    main()


# MLX TRAINING FILE CONVERSION

In [ ]:
import json, pathlib

src = pathlib.Path("sft.jsonl")
dst_dir = pathlib.Path("sft_mlx")
dst_dir.mkdir(exist_ok=True)

with open(dst_dir / "sft_mlx.jsonl", "w") as out:
    for line in src.open():
        obj = json.loads(line)
        out.write(json.dumps({
            "prompt": obj["instruction"],
            "completion": obj["response"]
        }) + "\n")


In [ ]:
import json
from pathlib import Path
import re

SRC = Path("data/sft_mlx/train.jsonl")
# SRC = Path("data/sft_mlx/valid.jsonl")
DST_DIR = Path("data/sft_mlx")
DST_DIR.mkdir(parents=True, exist_ok=True)

def clean(text):
    text = re.sub(r"<s>|</s>", "", text)
    text = re.sub(r"\[/?INST\]", "", text)
    return text.strip()

out = open(DST_DIR / "train.jsonl", "w")

for line in SRC.open():
    obj = json.loads(line)
    out.write(json.dumps({
        "prompt": clean(obj["instruction"]),
        "completion": clean(obj["response"]),
    }) + "\n")

out.close()
print("✅ MLX dataset cleaned and written")
